# Gemma 3n-E2B 日本語 LoRA ファインチューニング（破滅的忘却対策版）

> **注意**: 仕様書の「Gemma4-E2B」は **Gemma 3n-E2B** (`google/gemma-3n-E2B-it`) として実装しています。
> もし別のモデル（Gemma 3 4B など）を意図していた場合は、`model_name` 変数を変更してください。

## 📋 概要

- **対象モデル**: Gemma 3n-E2B (FP16)
- **フレームワーク**: Unsloth
- **データセット**: `llm-jp/oasst1-21k-ja` から抽出した高品質 8,000 件
- **手法**: LoRA（QLoRA ではない）
- **目的**: 日本語性能を底上げしつつ、破滅的忘却（catastrophic forgetting）を回避

## 🛡️ 破滅的忘却への対策

| # | 施策 | 本ノートブックでの設定 |
|---|------|----------------------|
| 1 | LoRA（フルチューンではない）| rank=16 / alpha=32 |
| 2 | 全 attention + MLP 層へ適用 | `q,k,v,o,gate,up,down_proj` |
| 3 | 学習率を高すぎない | `2e-4`（LoRA としては標準的） |
| 4 | エポックを絞る | 2 エポック（最大 3） |
| 5 | Cosine LR Schedule | 学習終盤で緩やかに減衰 |
| 6 | Warmup (5%) | 初期の不安定更新を抑制 |
| 7 | Weight Decay (0.01) | L2 正則化 |
| 8 | Gradient Checkpointing | メモリ節約 + 大きめバッチで安定化 |
| 9 | 学習前後の Perplexity 評価 | 忘却の定量モニタリング |

## ⚙️ ハイパーパラメータ

| 項目 | 値 |
|---|---|
| LoRA Rank | 16 |
| LoRA Alpha | 32 |
| Target modules | `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` |
| Gradient Accumulation Steps | 4 |
| Learning Rate | 2e-4 |
| Optimizer | `paged_adamw_8bit` |
| Epochs | 2 |
| Max Sequence Length | 2048 |
| Precision | FP16 |

## 🚀 実行環境

- Google Colab Pro 推奨（A100 / L4 なら 1 時間強で完了）
- T4 でも動作可能（ただし時間は 2 倍程度）


## Step 0: GPU 確認

Colab の GPU タイプを確認します。FP16 LoRA のため T4 (16GB) 以上が必要です。


In [ ]:
!nvidia-smi


## Step 1: ライブラリのインストール

Unsloth と関連パッケージをインストールします。Colab の Python 3.10+ 環境を想定しています。


In [ ]:
# Unsloth 本体（最新版）
!pip install --upgrade pip
!pip install "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"

# 依存パッケージ（バージョン競合を避けるため --no-deps で投入）
!pip install --no-deps "trl>=0.11" "peft>=0.11" accelerate bitsandbytes

# データセット / トークナイザ関連
!pip install -U datasets transformers

# 評価用
!pip install -U matplotlib seaborn


## Step 2: HuggingFace ログイン（Gemma は gated model）

Gemma は gated model のため、HuggingFace でライセンス同意済みのトークンが必要です。
[https://huggingface.co/google/gemma-3n-E2B-it](https://huggingface.co/google/gemma-3n-E2B-it) にアクセスし、事前にライセンス同意を行ってください。


In [ ]:
from huggingface_hub import login
import os

# Colab の Secret から読むか、直接文字列を指定
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Colab の Secrets に HF_TOKEN を登録しておくと自動取得
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("⚠️ HF_TOKEN が未設定です。Colab の Secrets に登録するか、login() を手動で呼んでください。")
    login()  # プロンプトで入力


## Step 3: インポート


In [ ]:
import os
import re
import random
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer
from trl import SFTTrainer
from unsloth import FastLanguageModel

sns.set_style("whitegrid")
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Step 4: モデル読み込み（FP16 / LoRA 用）

- **`dtype=torch.float16`**: FP16 指定
- **`load_in_4bit=False`**: QLoRA ではなく通常の LoRA なので 4bit 量子化しない
- **`max_seq_length=2048`**: 指定通り


In [ ]:
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/gemma-3n-E2B-it"  # Unsloth 最適化版（gated）

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,   # ★ FP16 指定
    load_in_4bit=False,    # ★ QLoRA ではなく LoRA
)

# Gemma のパディングトークン設定（念のため）
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## Step 5: LoRA 適用

指定通り **attention 全層 + MLP 全層** に LoRA を適用します。
- `r=16`, `alpha=32`（alpha = 2 * r の標準比率）
- `use_gradient_checkpointing="unsloth"`: VRAM 削減 + 学習安定化


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",       # attention
        "gate_proj", "up_proj", "down_proj",           # MLP
    ],
    lora_alpha=32,
    lora_dropout=0.05,        # ★ 忘却防止にドロップアウトを少し強め
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,         # rsLoRA は rank 64+ で効果的、16 ではオフ
    loftq_config=None,
)

# 学習可能パラメータ数を確認
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.3f}% of {total:,})")


## Step 6: データセット読み込み

`llm-jp/oasst1-21k-ja` を読み込みます。これは OpenAssistant の日本語翻訳データで、
約 21,000 件の対話サンプルが含まれます。


In [ ]:
raw_dataset = load_dataset("llm-jp/oasst1-21k-ja", split="train")
print(f"Total samples in dataset: {len(raw_dataset):,}")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nSample example:\n{raw_dataset[0]}")


## Step 7: 高品質サンプル抽出（8,000 件）

破滅的忘却を防ぐため、**質の高い多様な日本語サンプル**を抽出します。
以下の基準でフィルタリング:

1. **日本語含有率**: ひらがな/カタカナ/漢字の割合が 30% 以上
2. **応答長**: 50 文字以上 1,500 文字以下（長すぎず短すぎず）
3. **重複排除**: 同一 instruction は 1 件のみ
4. **ノイズ除去**: URLのみ/コードのみ/記号のみの応答を除外
5. **多様性確保**: シャッフル後に先頭 8,000 件

OASST1 に rank/score フィールドが存在する場合は、上位を優先的に採用します。


In [ ]:
JAPANESE_PATTERN = re.compile(r"[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]")
URL_ONLY_PATTERN = re.compile(r"^(https?://\S+\s*)+$")
CODE_ONLY_PATTERN = re.compile(r"^(\s*[{}\[\]();\n\s\w.+\-*/=<>!&|"'`]+$)", re.MULTILINE)


def japanese_ratio(text: str) -> float:
    """ひらがな/カタカナ/漢字の割合を返す"""
    if not text:
        return 0.0
    jp_chars = len(JAPANESE_PATTERN.findall(text))
    return jp_chars / max(len(text), 1)


def is_high_quality(example: dict) -> bool:
    """1 サンプルが高品質基準を満たすか判定"""
    # フィールド名の互換性対応（oasst1 は複数スキーマがあり得る）
    instruction = example.get("instruction") or example.get("input") or ""
    output = example.get("output") or example.get("response") or ""

    # 会話形式の場合は最初の human/assistant ペアを取り出す
    if not instruction and "conversations" in example:
        convs = example["conversations"]
        human_turns = [c for c in convs if c.get("from", "").lower() in ("human", "user")]
        gpt_turns = [c for c in convs if c.get("from", "").lower() in ("gpt", "assistant", "model")]
        if human_turns and gpt_turns:
            instruction = human_turns[0].get("value", "")
            output = gpt_turns[0].get("value", "")

    if not instruction or not output:
        return False

    # 長さフィルタ
    if len(output) < 50 or len(output) > 1500:
        return False
    if len(instruction) < 5 or len(instruction) > 1000:
        return False

    # 日本語含有率
    if japanese_ratio(output) < 0.30:
        return False
    if japanese_ratio(instruction) < 0.10:  # instruction は英語 OK だが日本語が少しはあること
        return False

    # ノイズ除去
    if URL_ONLY_PATTERN.match(output.strip()):
        return False

    return True


# フィルタリング実行
print("Filtering high-quality Japanese samples...")
filtered = []
seen_instructions = set()

for example in raw_dataset:
    if not is_high_quality(example):
        continue
    instr = example.get("instruction") or example.get("input") or ""
    if not instr and "conversations" in example:
        convs = example["conversations"]
        human_turns = [c for c in convs if c.get("from", "").lower() in ("human", "user")]
        if human_turns:
            instr = human_turns[0].get("value", "")
    if instr in seen_instructions:
        continue
    seen_instructions.add(instr)
    filtered.append(example)

print(f"After filtering: {len(filtered):,} samples")

# シャッフルして先頭 8,000 件
random.shuffle(filtered)
N_SAMPLES = 8000
if len(filtered) < N_SAMPLES:
    print(f"⚠️ フィルタ後サンプル数が {N_SAMPLES} に満たないため、全件使用します: {len(filtered)}")
    selected = filtered
else:
    selected = filtered[:N_SAMPLES]

print(f"Selected samples: {len(selected):,}")


## Step 8: データ正規化とプロンプトフォーマット

Gemma の公式チャットテンプレートを使用します。
Unsloth が内部で `tokenizer.apply_chat_template()` を呼ぶため、
`conversations` 形式のリストに正規化して SFTTrainer に渡します。


In [ ]:
def normalize_to_conversations(example: dict) -> list:
    """oasst1-ja の各レコードを [{role, content}, ...] 形式に正規化"""
    instruction = example.get("instruction") or example.get("input") or ""
    output = example.get("output") or example.get("response") or ""

    if not instruction and "conversations" in example:
        convs = example["conversations"]
        human_turns = [c for c in convs if c.get("from", "").lower() in ("human", "user")]
        gpt_turns = [c for c in convs if c.get("from", "").lower() in ("gpt", "assistant", "model")]
        if human_turns and gpt_turns:
            instruction = human_turns[0].get("value", "")
            output = gpt_turns[0].get("value", "")

    return [
        {"role": "user", "content": instruction.strip()},
        {"role": "assistant", "content": output.strip()},
    ]


# データセット化
normalized = [normalize_to_conversations(ex) for ex in selected]
train_dataset = Dataset.from_dict({"conversations": normalized})

# Gemma チャットテンプレートで1件確認
sample_text = tokenizer.apply_chat_template(
    train_dataset[0]["conversations"],
    tokenize=False,
    add_generation_prompt=False,
)
print("=" * 60)
print("Sample formatted text (first 800 chars):")
print("=" * 60)
print(sample_text[:800])
print("=" * 60)

# トークン長の分布を確認（学習時にフィルタされるが、目視用）
lengths = []
for ex in train_dataset.select(range(min(500, len(train_dataset)))):
    text = tokenizer.apply_chat_template(ex["conversations"], tokenize=False, add_generation_prompt=False)
    lengths.append(len(tokenizer.encode(text, add_special_tokens=False)))

plt.figure(figsize=(10, 4))
sns.histplot(lengths, bins=50)
plt.axvline(MAX_SEQ_LENGTH, color="red", linestyle="--", label=f"max_seq_length={MAX_SEQ_LENGTH}")
plt.title("Token length distribution (first 500 samples)")
plt.xlabel("Token length")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}, Max: {max(lengths)}, >2048: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}/{len(lengths)}")


## Step 9: 学習前評価（ベースライン Perplexity）

破滅的忘却を定量モニタリングするため、**学習前の Perplexity（日本語 / 英語）** を計測します。
学習後にもう一度計測し、日本語 PPL が下がりつつ英語 PPL が維持されていれば成功です。


In [ ]:
@torch.no_grad()
def compute_ppl(model, tokenizer, texts: list, max_length: int = 1024) -> float:
    """簡易 PPL 計算（バッチ処理）"""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    bs = 4
    for i in range(0, len(texts), bs):
        batch_texts = texts[i:i+bs]
        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(model.device)
        # labels = input_ids（padding は -100 でマスク）
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**enc, labels=labels)
        n_tokens = (labels != -100).sum().item()
        total_loss += out.loss.item() * n_tokens
        total_tokens += n_tokens
    avg_loss = total_loss / max(total_tokens, 1)
    return float(np.exp(avg_loss))


# 評価用サンプル（日本語 / 英語 各 50 件）
EVAL_JA_TEXTS = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": ex["conversations"][0]["content"]},
         {"role": "assistant", "content": ex["conversations"][1]["content"]}],
        tokenize=False, add_generation_prompt=False,
    )
    for ex in train_dataset.select(range(50))
]

EVAL_EN_TEXTS = [
    "<start_of_turn>user\nWhat is the capital of France?<end_of_turn>\n<start_of_turn>model\nThe capital of France is Paris.<end_of_turn>",
    "<start_of_turn>user\nExplain machine learning in one sentence.<end_of_turn>\n<start_of_turn>model\nMachine learning is a subset of artificial intelligence that enables systems to learn patterns from data.<end_of_turn>",
    "<start_of_turn>user\nWrite a Python function to reverse a string.<end_of_turn>\n<start_of_turn>model\ndef reverse_string(s):\n    return s[::-1]<end_of_turn>",
] * 17  # 51 件

print("Computing baseline PPL (this takes ~1 min)...")
ppl_ja_before = compute_ppl(model, tokenizer, EVAL_JA_TEXTS[:20])
ppl_en_before = compute_ppl(model, tokenizer, EVAL_EN_TEXTS[:20])
print(f"\n=== Baseline (before training) ===")
print(f"Japanese PPL: {ppl_ja_before:.3f}")
print(f"English  PPL: {ppl_en_before:.3f}")


## Step 10: 学習設定

指定ハイパーパラメータで `SFTTrainer` を構築します。


In [ ]:
# バッチサイズは GPU VRAM に応じて調整（A100 40GB: 4, T4 16GB: 1〜2）
PER_DEVICE_BATCH = 2 if torch.cuda.get_device_properties(0).total_memory > 20e9 else 1

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=2,                          # ★ エポック数
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=4,               # ★ 指定値
    learning_rate=2e-4,                          # ★ 指定値
    optim="paged_adamw_8bit",                    # ★ 指定値
    lr_scheduler_type="cosine",                  # 忘却防止: cosine
    warmup_ratio=0.05,                           # 5% warmup
    weight_decay=0.01,                           # L2 正則化
    max_grad_norm=1.0,
    fp16=True,                                   # ★ FP16
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,                          # 古い checkpoint 自動削除
    seed=3407,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,                 # SFTTrainer で会話形式を使う場合必要
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="conversations",          # チャット形式
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                               # oasst1 は短いサンプルが多いので packing はオフ
    args=training_args,
)

print(f"Effective batch size: {PER_DEVICE_BATCH} * 4 = {PER_DEVICE_BATCH * 4}")
print(f"Total optimization steps: {len(trainer.get_train_dataloader()) * 2 // (PER_DEVICE_BATCH * 4) + 1}")


## Step 11: 学習実行

約 1 時間強（A100） / 2-3 時間（T4） で完了します。


In [ ]:
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final loss: {trainer_stats.training_loss:.4f}")


## Step 12: 学習曲線の可視化

loss が急激に発散していないか、最後に持ち直していないかを確認します。


In [ ]:
log_history = trainer.state.log_history
losses = [entry["loss"] for entry in log_history if "loss" in entry]
steps = [entry["step"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, alpha=0.6, label="loss")
# 移動平均
if len(losses) > 10:
    window = 10
    ma = np.convolve(losses, np.ones(window)/window, mode="valid")
    plt.plot(steps[window-1:], ma, color="red", linewidth=2, label=f"MA({window})")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.tight_layout()
plt.show()


## Step 13: 学習後評価（Perplexity 比較）

日本語 PPL が下がり、英語 PPL が維持されていれば、破滅的忘却なく日本語能力が向上した証拠です。


In [ ]:
# LoRA アダプタをベースモデルに統合して評価（必要に応じて）
# ※ Unsloth は遅延マージを行うため、推論時は自動で適用されます

print("Computing post-training PPL...")
ppl_ja_after = compute_ppl(model, tokenizer, EVAL_JA_TEXTS[:20])
ppl_en_after = compute_ppl(model, tokenizer, EVAL_EN_TEXTS[:20])

print(f"\n=== PPL Comparison ===")
print(f"{'Metric':<15} {'Before':>10} {'After':>10} {'Δ':>10}")
print(f"{'-'*50}")
print(f"{'Japanese PPL':<15} {ppl_ja_before:>10.3f} {ppl_ja_after:>10.3f} {ppl_ja_after - ppl_ja_before:>+10.3f}")
print(f"{'English  PPL':<15} {ppl_en_before:>10.3f} {ppl_en_after:>10.3f} {ppl_en_after - ppl_en_before:>+10.3f}")
print()
if ppl_ja_after < ppl_ja_before:
    print("✅ 日本語 PPL が改善しました")
else:
    print("⚠️ 日本語 PPL が上昇しました（データ or LR を見直してください）")
if ppl_en_after - ppl_en_before < 1.0:
    print("✅ 英語 PPL は概ね維持されています（破滅的忘却なし）")
else:
    print("⚠️ 英語 PPL が上昇しました（破滅的忘却の可能性あり）")


## Step 14: 生成サンプル比較

いくつかの日本語プロンプトで生成品質を確認します。


In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # 推論モード

test_prompts = [
    "日本の四季について説明してください。",
    "Python でリストを逆順にする方法を教えてください。",
    "健康的な生活を送るための 3 つのアドバイスをお願いします。",
    "What is the difference between AI and machine learning?",
]

for prompt in test_prompts:
    print("=" * 70)
    print(f"USER: {prompt}")
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"MODEL: {response}")
    print()


## Step 15: モデル保存

### 15-1: LoRA アダプタのみ保存（軽量・推奨）


In [ ]:
# LoRA アダプタのみ（数十 MB）
ADAPTER_DIR = "./gemma3n-e2b-ja-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter saved to {ADAPTER_DIR}")

# Colab からダウンロード用
!ls -lh {ADAPTER_DIR}


### 15-2: （オプション）マージ済み 16bit モデルとして保存

ベースモデル + LoRA を統合した完全モデル（数 GB）を保存します。
HuggingFace Hub にアップロードする場合はこちらを推奨。


In [ ]:
# MERGED_DIR = "./gemma3n-e2b-ja-merged"
# model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
# print(f"✅ Merged 16bit model saved to {MERGED_DIR}")


### 15-3: （オプション）HuggingFace Hub へアップロード

学習したモデルを HuggingFace Hub に公開する場合。


In [ ]:
# from huggingface_hub import HfApi
# HF_REPO_ID = "your-username/gemma3n-e2b-ja-lora"  # ★ 書き換えてください
# 
# # マージ済みモデルをアップロード
# model.push_to_hub_merged(
#     HF_REPO_ID,
#     tokenizer,
#     save_method="merged_16bit",
#     token=HF_TOKEN,
# )


## Step 16: （オプション）GGUF 形式でエクスポート

llama.cpp / Ollama / LM Studio で使いたい場合。


In [ ]:
# GGUF_DIR = "./gemma3n-e2b-ja-gguf"
# model.save_pretrained_gguf(
#     GGUF_DIR,
#     tokenizer,
#     quantization_method=["q4_k_m", "q8_0"],  # 4bit と 8bit の両方
# )
# print(f"✅ GGUF saved to {GGUF_DIR}")


## Step 17: クリーンアップ & まとめ

学習結果のサマリを表示します。


In [ ]:
print("=" * 70)
print("Training Summary")
print("=" * 70)
print(f"Model:          {MODEL_NAME}")
print(f"Dataset:        llm-jp/oasst1-21k-ja (filtered to {len(selected)} samples)")
print(f"LoRA rank:      16 / alpha: 32")
print(f"Targets:        q,k,v,o,gate,up,down_proj")
print(f"LR:             2e-4 / optim: paged_adamw_8bit")
print(f"Epochs:         2")
print(f"Max seq len:    {MAX_SEQ_LENGTH}")
print(f"Training time:  {elapsed/60:.1f} min")
print(f"Final loss:     {trainer_stats.training_loss:.4f}")
print()
print(f"PPL (JA): {ppl_ja_before:.3f} -> {ppl_ja_after:.3f}")
print(f"PPL (EN): {ppl_en_before:.3f} -> {ppl_en_after:.3f}")
print("=" * 70)
